In [ ]:
import numpy as np
import os
import pandas as pd
import json
import pickle
from blablador import Models, Completions, ChatCompletions, TokenCount
import ast
import re
from itertools import chain
import country_converter as coco
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy
import time
import openai

In [ ]:
# Set your API key
openai.api_key = "sk-81sPgDrNQUVf7ixrHGbhuxDdYdxL43DwBaDGOxnHpaT3BlbkFJOT0lfm7nzaj3G1xFnRUFxyubARK0eHGKAXIjd5Q70A"

In [ ]:
df_labelled = pd.read_csv("validation_data/labelled_data.csv")

In [ ]:
def investigates_specific_events(text):
    prompt = f"""
    Context information is below.
    ---
    {text}
    ---
    Using only information from the text above, answer the query.
    Query: Does the text refer to a study that addresses one or more climate hazard events (i.e. it investigates the consequences of one or more events that happened in specific dates and locations)?
    Answer with 1 or 0. If yes, answer 1. If not, answer 0. Answer with either 1 or 0 and do not add extra text or notes.
    """
    return prompt

Functions to process the LLM response

In [ ]:
def extract_outer_json(text):
    start_index = text.find('{')
    end_index = text.rfind('}')

    if start_index == -1 or end_index == -1 or start_index >= end_index:
        return None  # Return None for empty JSON or invalid format

    extracted_json = text[start_index:end_index + 1]
    return extracted_json

In [ ]:
def run_llm(prompt, model_id="gpt-4o-mini"):
    try:

        response = openai.ChatCompletion.create(
            model=model_id,  # or "gpt-4" if you have access
            messages=[
            #{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
            ]
        )

        # Check if the expected keys are in the parsed data
        if "choices" in response and response["choices"] and "message" in response["choices"][0]:
            return response["choices"][0]["message"]["content"].strip()
        else:
            print("Error: Unexpected JSON structure")
            print("Parsed data:", response)
            return None

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
def add_key_value_pairs(data, new_pairs):
    """
    Adds new key-value pairs to each dictionary in a list of dictionaries.

    Parameters:
    data (list): A list of dictionaries.
    new_pairs (list or dict): A dictionary or list of dictionaries containing key-value pairs to be added.

    Returns:
    list: A list of dictionaries with the new key-value pairs added.
    """
    if isinstance(new_pairs, dict):
        # If new_pairs is a dictionary, add its key-value pairs to each dictionary in data
        for entry in data:
            for key, value in new_pairs.items():
                entry[key] = value
    elif isinstance(new_pairs, list):
        # If new_pairs is a list, add the corresponding dictionary's key-value pairs to the corresponding dictionary in data
        for entry, new_pair in zip(data, new_pairs):
            if isinstance(new_pair, dict):
                for key, value in new_pair.items():
                    entry[key] = value
    else:
        raise TypeError("new_pairs must be a dictionary or a list of dictionaries")
    
    return data

In [ ]:
results = []
response = []

In [ ]:
def get_event_information(df_labelled, model_id = 0):
    
    response = []
    count = 0
    for id, df in df_labelled.groupby("title"):
        
        print("Paper number: ", count)
        count += 1
        title = df["title"].iloc[0]
        abstract = df["abstract"].iloc[0]
        text = title + " " + abstract
        journal = df["journal"].values[0]
        doi = df["doi"].values[0]
        pmid = df["pmid"].values[0] if "pmid" in df else None
        source = df["source"].values[0] if "source" in df else None
        year = df["year"].values[0]

        reference_info = {
            "doi": doi,
            "pmid": pmid,
            "source": source,
            "journal": journal,
            "year": year,
            "title": title,
            "abstract": abstract
        }

        print(title)

        prompt = investigates_specific_events(text)
        result = run_llm(prompt, model_id)
        specific_event = int(''.join(re.findall("[01]", result)))

        print("Specific event: ", result)

        if specific_event == 1:
            
            hazard_types = check_hazard_type_keyword(text)

            for hazard in hazard_types:
                #hazard_type = hazard_description["hazardType"]
                #annotation = hazard_description["hazardAnnotation"]

                print("Hazard: ", hazard)
                
                prompt = check_event_occurrence(text, hazard)
                result = run_llm(prompt, model_id)
                hazard_occurrence = int(''.join(re.findall("[01]", result)))
                
                if hazard_occurrence == 1:
                
                    print("This hazard occurred in the study area.")
                    
                    prompt = get_event_location(text, hazard)
                    result = run_llm(prompt, model_id)
                    result_json = extract_outer_json(result)

                    try:
                        answer_location = json.loads(result_json.replace("\n", ""))["hazardLocation"]#json.loads("[" + result_json["hazardLocation"].replace("\n", "") + "]")
                    except Exception as e:
                        print("An unexpected error occurred:", e)
                        continue
                    if not answer_location:
                        print("JSON is empty:", result_json)
                        continue

                    if len(answer_location) > 1:
                        
                        for location in answer_location:
                            
                            # Check if 'country' key exists and has a value
                            if "country" in location and location["country"]:
                                location_country = location["country"]
                                print("Location: ", location_country)
                            else:
                                print("Country not specified or missing:", location)
                                continue
                            
                            # Check if 'city' key exists and has a value
                            if ("city" in location and location["city"]) or ("state" in location and location["state"]):
                                if location["city"]:
                                    location_city = location["city"]
                                    print("City: ", location_city)
                                    location_complete = f"{location_country}, {location_city}"
                                if location["state"]:
                                    location_state = location["state"]
                                    print("State: ", location_state)
                                    location_complete = f"{location_country}, {location_state}"
                            else:
                                print("City/State not specified or missing:", location)
                                location_complete = f"{location_country}"

                            prompt = get_event_date(text, hazard, location_complete)
                            result = run_llm(prompt, model_id)
                            result_json = extract_outer_json(result)

                            try:
                                answer_date = json.loads(result_json.replace("\n", ""))["hazardDate"]
                            except Exception as e:
                                print("An unexpected error occurred:", e)
                                continue
                            if not answer_date:
                                print("JSON is empty:", result_json)
                                continue
                                                        
                            data = add_key_value_pairs([{"hazardType": hazard, "specificEvent": specific_event}], location)
                            updated_data = deepcopy(add_key_value_pairs(data, answer_date))
                            updated_data = deepcopy(add_key_value_pairs(updated_data, reference_info))

                            response.append(deepcopy(updated_data))
                                            
                    else:
                        try:
                            location_country = answer_location[0]["country"]
                            print("Location: ", location_country)
                        except Exception as e:
                            print("An unexpected error occurred:", e)
                            continue
                        if not answer_location[0]["country"]:
                            print("Country not specified:", answer_location[0])
                            continue
                            
                        prompt = get_event_date(text, hazard, location_country)
                        result = run_llm(prompt, model_id)
                        result_json = extract_outer_json(result)

                        try:
                            answer_date = json.loads(result_json.replace("\n", ""))["hazardDate"]
                        except Exception as e:
                            print("An unexpected error occurred:", e)
                            print("Here is the result json:")
                            print(result_json)
                            continue
                        if not answer_date:
                            print("JSON is empty:", result_json)
                            continue
                                                    
                        data_unique_location = add_key_value_pairs([{"hazardType": hazard, "specificEvent": specific_event}], answer_location[0])
                        updated_data_unique_location = add_key_value_pairs(data_unique_location, answer_date)
                        updated_data_unique_location = add_key_value_pairs(updated_data_unique_location, reference_info)

                        response.append(updated_data_unique_location)

                        print(answer_date)
                else:
                    
                    print("This hazard did not occurr in the study area.")
                    
                    continue
                                            
        if specific_event == 0:
                
            print("Not a specific event")
            
            hazard_types = check_hazard_type_keyword(text)
            
            for hazard in hazard_types:
                            
                prompt = get_info_range(text, hazard)
                result = run_llm(prompt, model_id)
                result_json = extract_outer_json(result)
                                    
                try:
                    # Attempt to parse the answer as JSON
                    answer_parts = json.loads(result_json.replace("\n", ""))
                except json.JSONDecodeError:
                    print("Failed to parse JSON:", result_json)
                    continue  # Skip this iteration if parsing fails
                except Exception as e:
                    print("An unexpected error occurred:", e)
                    print("Here is the result json:")
                    print(result_json)
                    continue  # Skip this iteration if an unexpected error occurs
                if not answer_parts:  # If JSON is empty
                    print("JSON is empty:", result_json)
                    continue  # Skip this iteration if JSON is empty
                    
                print("Info: ", answer_parts)

                data_range = add_key_value_pairs([{"hazardType": hazard, "specificEvent": specific_event}], answer_parts)
                updated_data_range = add_key_value_pairs(data_range, reference_info)

                response.append(updated_data_range)
                time.sleep(1)
                
    response_unnested = list(chain(*response))
    response_df = pd.DataFrame(response_unnested)
    return(response, response_df)

In [ ]:
response, response_df = get_event_information(df_labelled, model_id="gpt-4o-mini")

Validation

In [ ]:
def calculate_precision(df1, df2, precision_columns_list):
    # Replace "nan" with empty string in df2 for specified columns
    for column in precision_columns_list:
        df2[column] = ["" if str(value) == "nan" else value for value in df2[column]]

    # Initialize results dictionary
    results = {col: {"psum": 0, "count": 0} for col in precision_columns_list}

    # Group by 'doi' and calculate precision for each column
    for id, df in df1.groupby("doi"):
        tmp = df2[df2["doi"] == id].reset_index(drop=True)
        tmp2 = df.reset_index(drop=True)

        if tmp.shape[0] == tmp2.shape[0]:
            for precision_column in precision_columns_list:
                precision_matches = (sorted(tmp[precision_column]) == sorted(tmp2[precision_column]))
                if precision_matches:
                    results[precision_column]["psum"] += 1
                results[precision_column]["count"] += 1

    # Calculate precision and create a DataFrame
    precision_values = []
    for precision_column in precision_columns_list:
        if results[precision_column]["count"] > 0:
            precision = results[precision_column]["psum"] / results[precision_column]["count"]
        else:
            precision = float('nan')  # Handle case where there is no data to calculate precision
        precision_values.append(precision)

    precision_df = pd.DataFrame([precision_values], columns=precision_columns_list)
    return precision_df